# Unsupervised Anomaly Detection

Isolation Forest · Local Outlier Factor · One-Class SVM
Trained on all data (or licit-only), evaluated on labeled subset.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from src.data_loader import load_all
from src.preprocessing import prepare_unsupervised
from src.models import isolation_forest, local_outlier_factor, one_class_svm, predict_anomaly, anomaly_scores
from src.evaluation import evaluate, plot_confusion_matrix, plot_pr_curve, plot_roc_curve

df, edges = load_all()
X_all, X_labeled, y_labeled, feat_cols, scaler = prepare_unsupervised(df)
print(f'All: {X_all.shape}  |  Labeled: {X_labeled.shape}  |  Illicit rate: {y_labeled.mean():.3f}')

## Isolation Forest

In [ ]:
iforest = isolation_forest(X_all, contamination=0.1)
y_pred_if = predict_anomaly(iforest, X_labeled)
y_score_if = anomaly_scores(iforest, X_labeled)

res_if = evaluate(y_labeled, y_pred_if, y_score_if, name='IsolationForest')
plot_confusion_matrix(y_labeled, y_pred_if, name='IsolationForest')

## Local Outlier Factor

In [ ]:
# LOF runs on labeled subset only (faster)
y_pred_lof, lof_model = local_outlier_factor(X_labeled, contamination=0.1)
res_lof = evaluate(y_labeled, y_pred_lof, name='LOF')
plot_confusion_matrix(y_labeled, y_pred_lof, name='LOF')

## One-Class SVM (trained on licit only)

In [ ]:
X_licit_only = X_labeled[y_labeled == 0]
ocsvm = one_class_svm(X_licit_only, nu=0.05)
y_pred_svm = predict_anomaly(ocsvm, X_labeled)
y_score_svm = anomaly_scores(ocsvm, X_labeled)

res_svm = evaluate(y_labeled, y_pred_svm, y_score_svm, name='OneClassSVM')
plot_confusion_matrix(y_labeled, y_pred_svm, name='OneClassSVM')

## Comparison

In [ ]:
import pandas as pd
results = pd.DataFrame([res_if, res_svm]).set_index('name')
print(results)

plot_pr_curve(y_labeled, {
    'IsolationForest': y_score_if,
    'OneClassSVM': y_score_svm
})